# Part 3: Frame-Autoregressive Pong

> **Learning objectives:**
> - Implement block-causal attention masks for temporal causality
> - Implement `modulate()` / `gate_fn()` for per-frame conditioning (the key change from Part 2)
> - Use them in CausalDiTBlock and CausalDiT forward passes
> - Sample videos autoregressively, one frame at a time

**Prerequisites:** Part 2 (DiT, flow matching on images).

### What changes from Part 2?

Only two things are conceptually new:

1. **Block-causal masking** — tokens in frame $t$ can attend to frames $\leq t$ but not $> t$
2. **Per-frame conditioning** — in Part 2, `cond` was `(B, d)` broadcast with `[:, None, :]`. Now `cond` is `(B, T, d)` and needs a reshape-broadcast-reshape pattern to reach each frame's tokens.

Everything else (Patch, UnPatch, Attention, GEGLU, RoPE, training loop) is provided.

In [ ]:
import math
import torch
import torch as t
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import sys
from pathlib import Path

exercises_dir = Path(".").resolve().parent
if str(exercises_dir) not in sys.path:
    sys.path.insert(0, str(exercises_dir))

from part3_far_pong import solutions
from part3_far_pong.tests import (
    test_block_causal_mask, test_modulate,
    test_causal_dit_block_forward,
    test_causal_dit_forward, test_causal_dit_state_dict_keys,
    test_sample_video,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Provided: All Building Blocks

The cells below give you everything from Part 2 (upgraded to match [toy-wm](https://github.com/wendlerc/toy-wm)) plus the new video-specific components. **You don't need to modify any of these** — just run them.

Data comes from [`pong_data.py`](pong_data.py): `get_pong_loader(...)` returns `(train_loader, pred2frame)` with batches of shape `(B, T, 3, 24, 24)` and `(B, T)`.

In [ ]:
import numpy as np
from part3_far_pong.pong_data import get_pong_loader

# ---- RMSNorm (toy-wm src/nn/norm.py) ----

class RMSNorm(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.w = nn.Parameter(t.ones(d))

    def forward(self, x):
        return x / ((x ** 2).mean(dim=-1, keepdim=True) + 1e-6).sqrt() * self.w


# ---- GEGLU FFN (toy-wm src/nn/geglu.py) ----

class GEGLU(nn.Module):
    def __init__(self, d_in, d_mid, d_out):
        super().__init__()
        self.up_proj = nn.Linear(d_in, d_mid, bias=True)
        self.up_proj.bias.data.zero_()
        self.up_gate = nn.Linear(d_in, d_mid, bias=True)
        self.up_gate.bias.data.zero_()
        self.down = nn.Linear(d_mid, d_out, bias=True)
        self.down.bias.data.zero_()
        self.nonlin = nn.SiLU()

    def forward(self, x):
        return self.down(self.up_proj(x) * self.nonlin(self.up_gate(x)))


# ---- Sinusoidal encoding (toy-wm src/nn/pe.py NumericEncoding) ----

class NumericEncoding(nn.Module):
    def __init__(self, C=5000, dim=64, n_max=1000):
        super().__init__()
        args = t.exp(-math.log(C) * t.arange(0, dim, 2) / dim)
        args = t.arange(n_max)[:, None] * args[None, :]
        pe = t.empty((n_max, dim))
        pe[:, ::2] = t.sin(args)
        pe[:, 1::2] = t.cos(args)
        self.register_buffer("pe", pe)

    def forward(self, num):
        return self.pe[num]


# ---- Rotary Position Embeddings (toy-wm src/nn/pe.py RoPE) ----

class RoPE(nn.Module):
    def __init__(self, d_head, n_ctx, C=10000):
        super().__init__()
        thetas = t.exp(-math.log(C) * t.arange(0, d_head, 2) / d_head)
        thetas = thetas.repeat(2, 1).T.flatten()
        positions = t.arange(n_ctx)
        all_thetas = positions.unsqueeze(1) * thetas.unsqueeze(0)
        self.register_buffer('sins', t.sin(all_thetas).unsqueeze(0).unsqueeze(2))
        self.register_buffer('coss', t.cos(all_thetas).unsqueeze(0).unsqueeze(2))

    def forward(self, x, offset=0):
        x_perm = t.empty_like(x)
        even = t.arange(0, x.shape[-1], 2, device=x.device)
        odd = t.arange(1, x.shape[-1], 2, device=x.device)
        x_perm[..., even] = -x[..., odd]
        x_perm[..., odd] = x[..., even]
        return (self.coss[:, offset:offset + x.shape[1]] * x
                + self.sins[:, offset:offset + x.shape[1]] * x_perm)


# ---- Patch / UnPatch (toy-wm src/nn/patch.py) ----

class Patch(nn.Module):
    def __init__(self, in_channels=3, out_channels=64, patch_size=2):
        super().__init__()
        self.patch_size = patch_size
        self.out_channels = out_channels
        dim = out_channels
        if dim % 32 == 0 and dim > 32:
            self.init_conv_seq = nn.Sequential(
                nn.Conv2d(in_channels, dim // 2, kernel_size=5, padding=2, stride=1),
                nn.SiLU(), nn.GroupNorm(32, dim // 2),
                nn.Conv2d(dim // 2, dim // 2, kernel_size=5, padding=2, stride=1),
                nn.SiLU(), nn.GroupNorm(32, dim // 2),
            )
        else:
            self.init_conv_seq = nn.Sequential(
                nn.Conv2d(in_channels, dim // 2, kernel_size=5, padding=2, stride=1), nn.SiLU(),
                nn.Conv2d(dim // 2, dim // 2, kernel_size=5, padding=2, stride=1), nn.SiLU(),
            )
        self.x_embedder = nn.Linear(patch_size * patch_size * dim // 2, dim, bias=True)
        nn.init.constant_(self.x_embedder.bias, 0)

    def patchify(self, x):
        B, C, H, W = x.size()
        ps = self.patch_size
        x = x.view(B, C, H // ps, ps, W // ps, ps)
        return x.permute(0, 2, 4, 1, 3, 5).flatten(-3).flatten(1, 2)

    def forward(self, x):
        batch, dur, c, h, w = x.shape
        x = self.init_conv_seq(x.reshape(-1, c, h, w))
        x = self.x_embedder(self.patchify(x))
        return x.reshape(batch, dur, -1, self.out_channels)


class UnPatch(nn.Module):
    def __init__(self, height, width, in_channels=64, out_channels=3, patch_size=2):
        super().__init__()
        self.width, self.height = width, height
        self.patch_size, self.out_channels = patch_size, out_channels
        self.unpatch = nn.Linear(in_channels, out_channels * patch_size ** 2)

    def forward(self, x):
        batch, dur, seq, d = x.shape
        x = self.unpatch(x).reshape(-1, seq, self.out_channels * self.patch_size ** 2)
        c, p = self.out_channels, self.patch_size
        h, w = self.height // p, self.width // p
        x = t.einsum("nhwpqc->nchpwq", x.reshape(-1, h, w, p, p, c))
        return x.reshape(batch, dur, c, h * p, w * p)


# ---- Attention with QK-norm, RoPE, and block-causal mask ----

class CausalVideoAttention(nn.Module):
    def __init__(self, d=64, n_head=4, rope=None):
        super().__init__()
        self.n_head, self.d, self.d_head = n_head, d, d // n_head
        self.QKV = nn.Linear(d, 3 * d)
        self.O = nn.Linear(d, d)
        self.lnq = RMSNorm(self.d_head)
        self.lnk = RMSNorm(self.d_head)
        self.rope = rope

    def forward(self, x, mask=None):
        b, s, d = x.shape
        q, k, v = self.QKV(x).chunk(3, dim=-1)
        q = self.lnq(q.reshape(b, s, self.n_head, self.d_head))
        k = self.lnk(k.reshape(b, s, self.n_head, self.d_head))
        v = v.reshape(b, s, self.n_head, self.d_head)
        if self.rope is not None:
            q, k = self.rope(q), self.rope(k)
        q, k, v = [t_.permute(0, 2, 1, 3) for t_ in (q, k, v)]
        attn = q @ k.transpose(-2, -1)
        if mask is not None:
            attn = attn.masked_fill(mask[None, None, :, :], float('-inf'))
        z = attn.softmax(dim=-1) @ v
        return self.O(z.permute(0, 2, 1, 3).reshape(b, s, d))

## Exercise 2 — `modulate()` and `gate_fn()`
*Difficulty: 🔴🔴🔴⭕⭕ | ~15 min*

This is **the** key change from Part 2. In Part 2, conditioning was per-sample `(B, d)` and you broadcast it to tokens with `[:, None, :]`:

```python
# Part 2: cond is (B, d), x is (B, seq, d)
x = norm(x) * (1 + scale[:, None, :]) + bias[:, None, :]
x = attn(x) * gate[:, None, :]
```

For video, conditioning is **per-frame** `(B, T, d)`, but the token sequence is `(B, T * toks_per_frame, d)`. Implement `modulate(x, shift, scale)` for affine modulation and `gate_fn(x, g)` for gating. 

## Recap & Motivation

We are going to patchify videos similar like we did with the images in MNIST. Now instead of having a single conditioning signal, we have one for each frame. Each frame is conditioned on the frame specific noise level (flow matching) and the action of the previous frame. 

To avoid applying all layers acting upon our conditioning vectors for each patch in each frame we are going to implement some broadcasting utilities here.

In [ ]:
from IPython.display import HTML

with open("patchify_diagram.html") as f:
    display(HTML(f.read()))

## Visual task description

In [ ]:
from IPython.display import HTML

with open("modulation_diagram.html") as f:
    display(HTML(f.read()))

In [ ]:
def modulate(x, shift, scale):
    """Per-frame affine modulation.

    Args:
        x:     (B, S, d) where S = n_frames * toks_per_frame
        shift: (B, n_frames, d) per-frame shift
        scale: (B, n_frames, d) per-frame scale

    Returns:
        (B, S, d) — x * (1 + scale) + shift, broadcast per frame
    """
    # YOUR CODE HERE
    B, n_frames, d = shift.shape
    x = x.reshape(B, n_frames, -1, d)
    x = x * (1 + scale[:, :, None, :]) + shift[:, :, None, :]
    return x.reshape(B, -1, d)


def gate_fn(x, g):
    """Per-frame gating.

    Args:
        x: (B, S, d) where S = T * toks_per_frame
        g: (B, T, d) per-frame gate

    Returns:
        (B, S, d) — x * g, broadcast per frame
    """
    # YOUR CODE HERE
    B, n_frames, d = g.shape
    x = x.reshape(B, n_frames, -1, d)
    x = x * g[:, :, None, :]
    return x.reshape(B, -1, d)

In [ ]:
test_modulate(modulate, gate_fn)

In [ ]:
# Visualize some preprocessed data
_viz_loader, _pred2frame = get_pong_loader(batch_size=1, shuffle=False, num_workers=0, duration=16, fps=1)
frames, actions = next(iter(_viz_loader))
print(frames.shape)
print(actions.shape)
print(f"One episode: frames {frames.shape}, actions {actions.shape}")

# Only handle the case frames is shape (1, 16, 3, 24, 24)
# So single batch, 16 frames, 3 channels, 24x24
assert frames.shape == (1, 16, 3, 24, 24)
frames = frames[0]
actions = actions[0]
action_names = {1: "noop", 2: "up", 3: "down"}
fig, axes = plt.subplots(1, 8, figsize=(16, 2))

for i in range(8):
    ax = axes[i]
    frame = frames[i]  # (3, 24, 24)
    img = (frame.permute(1, 2, 0).numpy() + 1) / 2  # (24, 24, 3), scaled [0, 1]
    ax.imshow(img.clip(0, 1))
    a = actions[i].item()
    ax.set_title(f"a={a} ({action_names.get(a, '?')})")
    ax.axis('off')

plt.suptitle("First 8 frames — preprocessed episode")
plt.tight_layout()
plt.show()

In [ ]:
# Visualize a composite image of the last 2 columns from each of 50 frames,
# with action annotations below the composite image (one per frame).

_viz_loader, _pred2frame = get_pong_loader(batch_size=1, shuffle=False, num_workers=0, fps=1, duration=50)
print(_viz_loader)
episode_frames, episode_actions = _viz_loader.dataset[0]
print(episode_frames.shape)
print(f"One episode: frames {episode_frames.shape}, actions {episode_actions.shape}")

num_frames = 50
cols_to_take = 2  # last 2 columns
H, W = episode_frames.shape[1], episode_frames.shape[2]

# Stack the last 2 columns of each frame to construct final image (H, cols_to_take*num_frames, C)
last2_cols = []
for i in range(num_frames):
    frame = episode_frames[i]  # (C, H, W)
    patch = frame[:, :, -cols_to_take:]  # (C, H, 2)
    last2_cols.append(patch.permute(1, 2, 0).numpy())  # to (H, 2, C)
last2_cols = [((p + 1) / 2).clip(0, 1) for p in last2_cols]  # scale to [0, 1]

# Concatenate along width
composite = np.concatenate(last2_cols, axis=1)  # (H, cols_to_take*num_frames, C)

action_names = {1: "-", 2: "u", 3: "d"}
action_labels = [action_names.get(episode_actions[i].item(), "?") for i in range(num_frames)]

fig, ax = plt.subplots(figsize=(num_frames * 0.3, 4))
ax.imshow(composite)
ax.axis("off")
ax.set_title("Last 2 columns of each of the first 50 frames")

# Add action labels under each 2-column patch
xtick_positions = np.arange(num_frames) * cols_to_take + cols_to_take / 2

for i, x in enumerate(xtick_positions):
    ax.text(x, H + 4, action_labels[i], ha='center', va='top', fontsize=8, rotation=0)

plt.ylim((composite.shape[0] + 20, -5))  # Make space for labels below
plt.tight_layout()
plt.show()

## Exercise 1 — Block-Causal Attention Mask
*Difficulty: 🔴🔴⭕⭕⭕ | ~5 min*

Create a boolean mask where `True` = blocked. Tokens in frame $t$ can attend to all tokens in frames $\leq t$ but not to any token in frames $> t$.

In [ ]:
from IPython.display import display, HTML

display(HTML('''
<div style="background: #2b2b2b; padding: 20px; border-radius: 8px; display: inline-block;">
  <img src="df_masking.jpg" width="600"/>
</div>
'''))

In [ ]:
def make_block_causal_mask(n_frames, toks_per_frame, device="cpu"):
    """Create block-causal mask. True = blocked (masked out).

    Args:
        n_frames: number of video frames
        toks_per_frame: number of tokens per frame (patches + registers)

    Returns:
        bool tensor of shape (total_tokens, total_tokens)
    """
    return (1-t.kron(t.tril(torch.ones(n_frames, n_frames, device=device)), torch.ones(toks_per_frame, toks_per_frame, device=device))).bool()
    

In [ ]:
test_block_causal_mask(make_block_causal_mask)

## Exercise 3 — CausalDiTBlock.forward
*Difficulty: 🔴🔴⭕⭕⭕ | ~5 min*

The structure is identical to Part 2’s DiTBlock — just use your `modulate()` and `gate_fn()` on the 4 marked lines.

In [ ]:
class CausalDiTBlock(nn.Module):
    """DiT block with causal video attention and per-frame modulation."""
    def __init__(self, d=64, n_head=4, exp=4, rope=None):
        super().__init__()
        self.norm1 = RMSNorm(d)
        self.selfattn = CausalVideoAttention(d, n_head, rope=rope)
        self.norm2 = RMSNorm(d)
        self.geglu = GEGLU(d, exp * d, d)
        self.modulation = nn.Sequential(nn.SiLU(), nn.Linear(d, 6 * d, bias=True))

    def forward(self, x, cond, mask=None):
        mu1, sigma1, c1, mu2, sigma2, c2 = self.modulation(cond).chunk(6, dim=-1)
        residual = x
        x = modulate(self.norm1(x), mu1, sigma1)
        x = self.selfattn(x, mask=mask)
        x = residual + gate_fn(x, c1)
        residual = x
        x = modulate(self.norm2(x), mu2, sigma2)
        x = self.geglu(x)
        x = residual + gate_fn(x, c2)
        return x

In [ ]:
test_causal_dit_block_forward(CausalDiTBlock, make_block_causal_mask)

## Provided — CausalDiT.forward

In [ ]:
class CausalDiT(nn.Module):
    def __init__(self, h=24, w=24, n_actions=4, in_channels=3,
                 patch_size=3, n_blocks=8, d=320, n_head=20, exp=4,
                 T=1000, C=5000, n_registers=1, n_window=30):
        super().__init__()
        self.T = T
        self.n_blocks = n_blocks
        self.n_registers = n_registers
        self.n_window = n_window
        self.in_channels = in_channels
        self.toks_per_frame = (h // patch_size) * (w // patch_size) + n_registers
        self.patches_per_frame = self.toks_per_frame
        d_head = d // n_head
        rope_ctx = n_window * self.toks_per_frame
        self.rope_seq = RoPE(d_head, rope_ctx, C=C)

        self.blocks = nn.ModuleList(
            [CausalDiTBlock(d, n_head, exp, rope=self.rope_seq) for _ in range(n_blocks)]
        )
        self.patch = Patch(in_channels, d, patch_size)
        self.norm = RMSNorm(d)
        self.unpatch = UnPatch(h, w, d, in_channels, patch_size)
        self.action_emb = nn.Embedding(n_actions, d)
        self.registers = nn.Parameter(t.randn(n_registers, d) * d ** -0.5)
        self.time_emb = NumericEncoding(C=C, dim=d, n_max=T)
        self.time_emb_mixer = nn.Linear(d, d)
        self.modulation = nn.Sequential(nn.SiLU(), nn.Linear(d, 2 * d, bias=True))

    def tokenize(self, x):
        """(B, T, C, H, W) -> (B, T*toks_per_frame, d)"""
        B, T_frames = x.shape[:2]
        z = self.patch(x)
        regs = self.registers[None, None].expand(B, T_frames, -1, -1)
        zr = t.cat([z, regs], dim=2)
        self._dur, self._seq = zr.shape[1], zr.shape[2]
        return zr.reshape(B, -1, zr.shape[-1])

    def detokenize(self, zr):
        """(B, T*toks_per_frame, d) -> (B, T, C, H, W)"""
        B = zr.shape[0]
        zr = zr.reshape(B, self._dur, self._seq, -1)
        return self.unpatch(zr[:, :, :-self.n_registers])

    def forward(self, x, actions, ts):
        B, T_frames, C, H, W = x.shape

        # per-frame conditioning (B, T, d)
        a = self.action_emb(actions)
        ts_scaled = (ts.float() * (self.T - 1)).long()
        cond = self.time_emb_mixer(self.time_emb(ts_scaled)) + a

        zr = self.tokenize(x)
        mask = make_block_causal_mask(T_frames, self.toks_per_frame, device=x.device)

        for block in self.blocks:
            zr = block(zr, cond, mask=mask)

        mu, sigma = self.modulation(cond).chunk(2, dim=-1)
        zr = modulate(self.norm(zr), mu, sigma)

        return self.detokenize(zr)

    @property
    def device(self):
        return self.registers.device

In [ ]:
test_causal_dit_forward(CausalDiT)
test_causal_dit_state_dict_keys(CausalDiT)

## Provided: Training with Diffusion Forcing

Same flow matching loss as Part 2, but with per-frame timesteps `(B, T)` instead of per-sample `(B,)`, and action dropout instead of class dropout. This is provided since the changes from Part 2 are minimal.

In [ ]:
from muon import SingleDeviceMuonWithAuxAdam

def get_muon(model, lr1=0.02, lr2=3e-4, betas=(0.9, 0.95), weight_decay=1e-5):
    body_weights = list(model.blocks.parameters())
    body_ids = {id(p) for p in body_weights}
    other_weights = [p for p in model.parameters() if id(p) not in body_ids]
    hidden_weights = [p for p in body_weights if p.ndim >= 2]
    hidden_gains_biases = [p for p in body_weights if p.ndim < 2]
    return SingleDeviceMuonWithAuxAdam([
        dict(params=hidden_weights, use_muon=True, lr=lr1, weight_decay=weight_decay),
        dict(params=hidden_gains_biases + other_weights, use_muon=False,
             lr=lr2, betas=betas, weight_decay=weight_decay),
    ])


def train_pong_model(model, train_loader, n_steps=2500, lr1=0.02, lr2=3e-4, action_dropout=0.2):
    """Diffusion forcing training. Same as Part 2 but per-frame timesteps."""
    optimizer = get_muon(model, lr1=lr1, lr2=lr2)
    running_loss, train_iter = 0, iter(train_loader)

    for step in range(n_steps):
        try:
            frames, actions = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            frames, actions = next(train_iter)

        frames = frames[:, :model.n_window].to(device)
        actions = actions[:, :model.n_window].to(device)
        B, T_frames = frames.shape[:2]

        ts = F.sigmoid(t.randn(B, T_frames, device=device))        # per-frame timestep
        z = t.randn_like(frames)
        v_true = frames - z
        x_t = frames - ts[:, :, None, None, None] * v_true

        actions = actions.clone()
        actions[t.rand(B, T_frames, device=device) < action_dropout] = 0  # action dropout for CFG

        loss = F.mse_loss(model(x_t, actions, ts), v_true)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        optimizer.step()

        running_loss += loss.item()
        if (step + 1) % 100 == 0:
            print(f"Step {step+1}/{n_steps} | Loss: {running_loss / 100:.4f}")
            running_loss = 0


# Load data and create model
train_loader, pred2frame = get_pong_loader(batch_size=8, shuffle=True, num_workers=4, duration=16, fps=1)
dataset = train_loader.dataset
print(f"Dataset: {len(dataset)} episodes")

model = CausalDiT(h=24, w=24, n_actions=4, in_channels=3,
                   patch_size=4, n_blocks=8, d=512, n_head=32, exp=2,
                   T=1000, C=5000, n_registers=1, n_window=16).to(device)
print(f"Model: {sum(p.numel() for p in model.parameters()):,} parameters")

train_pong_model(model, train_loader, n_steps=2500)

## Exercise 5 — Video Sampling
*Difficulty: 🔴🔴🔴🔴⭕ | ~20 min*

This is genuinely new compared to Part 2. Generate video **autoregressively**, one frame at a time:

1. Start with the first frame (given).
2. For each new frame: start from noise `z`, denoise it over `n_denoise_steps` using the SD3 schedule.
3. At each denoising step, concatenate all previous **clean** frames (ts=0) with the current **noisy** frame (ts=t_i) and run the full model. Extract the velocity prediction for the **last frame only**.
4. After denoising, append the clean frame to the video.

For CFG: run the model twice (once with real actions, once with zeroed actions) and interpolate.

In [ ]:
@t.no_grad()
def sample_video(model, first_frame, actions, n_denoise_steps=30, cfg=0):
    """Generate video autoregressively, one frame at a time."""
    B = first_frame.shape[0]
    total_frames = actions.shape[1]
    C, H, W = first_frame.shape[2:]
    n_window = getattr(model, 'n_window', total_frames)

    video = first_frame.clone()

    for frame_idx in range(1, total_frames):
        z = t.randn(B, 1, C, H, W, device=first_frame.device, dtype=first_frame.dtype)

        denoise_ts = t.linspace(1, 0, n_denoise_steps + 1, device=first_frame.device)
        denoise_ts = 3 * denoise_ts / (2 * denoise_ts + 1)

        for step_idx in range(n_denoise_steps):
            current_video = t.cat([video, z], dim=1)
            if current_video.shape[1] > n_window:
                current_video = current_video[:, -n_window:]
            n_ctx = current_video.shape[1]

            ts = t.zeros(B, n_ctx, device=first_frame.device)
            ts[:, -1] = denoise_ts[step_idx]

            act = actions[:, frame_idx + 1 - n_ctx:frame_idx + 1]

            v_pred = model(current_video, act, ts)
            if cfg > 0:
                v_uncond = model(current_video, act * 0, ts)
                v_pred = v_uncond + cfg * (v_pred - v_uncond)

            dt = denoise_ts[step_idx] - denoise_ts[step_idx + 1]
            z = z + dt * v_pred[:, -1:]

        video = t.cat([video, z], dim=1)

    return video

In [ ]:
test_sample_video(sample_video, CausalDiT)

In [ ]:
import numpy as np
from matplotlib import animation
from IPython.display import HTML

model.eval()
idx = 42
first_frame = dataset[idx][0][:1].unsqueeze(0).to(device)
all_actions = t.full((1, 30), 2, device=device, dtype=t.long)  # 30 frames of "up"

video = sample_video(model, first_frame, all_actions, n_denoise_steps=20, cfg=2.0)
video_np = (video[0].cpu().permute(0, 2, 3, 1).clamp(-1, 1) * 0.5 + 0.5).numpy()

fig, ax = plt.subplots(figsize=(3.6, 3.6))
im = ax.imshow(video_np[0]); ax.axis('off')
def update(f): im.set_array(video_np[f]); ax.set_title(f"Frame {f}"); return [im]
anim = animation.FuncAnimation(fig, update, frames=len(video_np), interval=167, blit=True)
plt.close()
HTML(anim.to_jshtml())

## Summary

The two key changes from Part 2:
1. **Block-causal masking** prevents future-frame attention
2. **Per-frame conditioning** via `modulate()` / `gate_fn()` replaces the `[:, None, :]` broadcast

Everything else (architecture, loss, optimizer) carried over with minimal changes.

Next: **Part 4** adds KV-caching to make autoregressive sampling efficient.